# NB03 — Baseline Models on the Original 11 Morphometric Features

Leakage-safe nested stratified CV for baseline classifiers. Hyperparameters are selected only within inner training folds.

**Corrected version.** The target is numerically encoded for all classifiers to avoid the scikit-learn MLP early-stopping error observed with string labels in the current Colab environment. Checkpoints are saved after every model/seed.


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, random, warnings, hashlib, platform, sys
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ROOT = Path(r"/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1")
DATA = ROOT / "01_DATA"
NOTEBOOKS = ROOT / "02_NOTEBOOKS"
RESULTS = ROOT / "03_RESULTS"
MODELS = ROOT / "04_MODELS"
FIGURES = ROOT / "05_FIGURES"
TABLES = ROOT / "06_TABLES"
LOGS = ROOT / "07_LOGS"
EXPORTS = ROOT / "08_EXPORTS"
FINAL_ZIP = ROOT / "09_FINAL_ZIP"

for p in [DATA, NOTEBOOKS, RESULTS, MODELS, FIGURES, TABLES, LOGS, EXPORTS, FINAL_ZIP]:
    p.mkdir(parents=True, exist_ok=True)

DATASET = DATA / "INIAP_Dataset.xlsx"
SEEDS = [2026, 2027, 2028]
OUTER_FOLDS = 5
INNER_FOLDS = 3
TARGET = "Class"

assert DATASET.exists(), f"Dataset not found: {DATASET}"
print("ROOT:", ROOT)
print("DATASET:", DATASET)


In [ ]:

!pip -q install xgboost


In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


In [ ]:

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    f1_score, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, matthews_corrcoef, cohen_kappa_score,
    log_loss
)
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline

def metrics_dict(y_true, y_pred, y_prob=None):
    d = {
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "kappa": cohen_kappa_score(y_true, y_pred)
    }
    if y_prob is not None:
        try:
            d["log_loss"] = log_loss(y_true, y_prob)
        except Exception:
            d["log_loss"] = np.nan
    return d

def nested_cv_evaluate(name, estimator, param_grid, X, y, seed):
    outer = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=seed)
    rows, preds, params = [], [], []

    for fold, (tr, te) in enumerate(outer.split(X, y), start=1):
        print(f"  {name} | seed {seed} | outer fold {fold}/{OUTER_FOLDS}", flush=True)
        inner = StratifiedKFold(
            n_splits=INNER_FOLDS, shuffle=True, random_state=seed + fold
        )
        search = GridSearchCV(
            estimator,
            param_grid=param_grid,
            scoring="f1_macro",
            cv=inner,
            n_jobs=-1,
            refit=True,
            return_train_score=False,
            error_score="raise"
        )
        search.fit(X.iloc[tr], y.iloc[tr])
        best = search.best_estimator_
        pred = best.predict(X.iloc[te])
        prob = best.predict_proba(X.iloc[te]) if hasattr(best, "predict_proba") else None

        met = metrics_dict(y.iloc[te], pred, prob)
        met.update({
            "model": name, "seed": seed,
            "outer_fold": fold, "n_test": len(te)
        })
        rows.append(met)
        params.append({
            "model": name, "seed": seed, "outer_fold": fold,
            "best_score_inner": search.best_score_,
            "best_params": json.dumps(search.best_params_)
        })

        for j, idx in enumerate(te):
            rec = {
                "row_id": int(idx),
                "y_true": int(y.iloc[idx]),
                "y_pred": int(pred[j]),
                "model": name,
                "seed": seed,
                "outer_fold": fold
            }
            if prob is not None:
                for k, cls in enumerate(best.classes_):
                    rec[f"prob_{int(cls)}"] = float(prob[j, k])
            preds.append(rec)

    return pd.DataFrame(rows), pd.DataFrame(preds), pd.DataFrame(params)


In [ ]:

OUT = RESULTS / "NB03_BASELINES"; OUT.mkdir(exist_ok=True)
TAB = TABLES / "NB03_BASELINES"; TAB.mkdir(exist_ok=True)
LOG = LOGS / "NB03_BASELINES"; LOG.mkdir(exist_ok=True)

df = pd.read_excel(DATASET).rename(columns={"AspectRation": "AspectRatio"})
X = df.drop(columns=[TARGET]).copy()
y_text = df[TARGET].astype(str).copy()

# Encode target ONCE for every classifier. This avoids the sklearn 1.6 / Python 3.13
# MLP early-stopping incompatibility with string labels and keeps all models comparable.
le = LabelEncoder()
y = pd.Series(le.fit_transform(y_text), index=df.index, name=TARGET)
CLASS_MAP = {int(i): str(cls) for i, cls in enumerate(le.classes_)}
print("Class encoding:", CLASS_MAP)

models = {
    "LogisticRegression": (
        Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(max_iter=5000))
        ]),
        {"clf__C": [0.1, 1, 10]}
    ),
    "SVM_RBF": (
        Pipeline([
            ("scale", StandardScaler()),
            ("clf", SVC(kernel="rbf", probability=True))
        ]),
        {"clf__C": [1, 10, 100], "clf__gamma": ["scale", 0.01, 0.1]}
    ),
    "XGBoost": (
        XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            n_jobs=1,
            tree_method="hist"
        ),
        {
            "n_estimators": [300, 600],
            "max_depth": [3, 5],
            "learning_rate": [0.03, 0.1],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0]
        }
    ),
    "MLP": (
        Pipeline([
            ("scale", StandardScaler()),
            ("clf", MLPClassifier(max_iter=1200, early_stopping=True))
        ]),
        {
            "clf__hidden_layer_sizes": [(64,), (128, 64)],
            "clf__alpha": [1e-4, 1e-3],
            "clf__learning_rate_init": [3e-4, 1e-3]
        }
    )
}


In [ ]:

all_metrics, all_preds, all_params = [], [], []

for seed in SEEDS:
    for name, (est, grid) in models.items():
        print(f"\n=== Running {name} | seed {seed} ===", flush=True)

        # Fresh clone for every model/seed and explicit random state when supported.
        est_seed = clone(est)
        if name == "XGBoost":
            est_seed.set_params(random_state=seed)
        else:
            try:
                est_seed.set_params(clf__random_state=seed)
            except ValueError:
                pass

        m, p, b = nested_cv_evaluate(name, est_seed, grid, X, y, seed)
        all_metrics.append(m)
        all_preds.append(p)
        all_params.append(b)

        # Checkpoints: preserve completed work after every model/seed.
        pd.concat(all_metrics, ignore_index=True).to_csv(
            OUT / "baseline_metrics_CHECKPOINT.csv", index=False
        )
        pd.concat(all_preds, ignore_index=True).to_csv(
            OUT / "baseline_oof_predictions_CHECKPOINT.csv", index=False
        )
        pd.concat(all_params, ignore_index=True).to_csv(
            OUT / "baseline_best_params_CHECKPOINT.csv", index=False
        )
        print(f"Completed {name} | seed {seed}. Checkpoint saved.", flush=True)

metrics = pd.concat(all_metrics, ignore_index=True)
preds = pd.concat(all_preds, ignore_index=True)
params = pd.concat(all_params, ignore_index=True)

# Decode target labels only for human-readable saved predictions.
preds["y_true"] = preds["y_true"].astype(int).map(CLASS_MAP)
preds["y_pred"] = preds["y_pred"].astype(int).map(CLASS_MAP)
preds = preds.rename(columns={
    f"prob_{i}": f"prob_{cls}" for i, cls in CLASS_MAP.items()
})

metrics.to_csv(OUT / "baseline_metrics_by_fold.csv", index=False)
preds.to_csv(OUT / "baseline_oof_predictions.csv", index=False)
params.to_csv(OUT / "baseline_best_params.csv", index=False)

summary = metrics.groupby("model").agg(
    mean_macro_f1=("f1_macro", "mean"),
    sd_macro_f1=("f1_macro", "std"),
    mean_accuracy=("accuracy", "mean"),
    mean_balanced_accuracy=("balanced_accuracy", "mean"),
    mean_mcc=("mcc", "mean"),
    mean_kappa=("kappa", "mean")
).sort_values("mean_macro_f1", ascending=False)

summary.to_csv(TAB / "baseline_summary.csv")

run_info = {
    "target_encoding": CLASS_MAP,
    "seeds": SEEDS,
    "outer_folds": OUTER_FOLDS,
    "inner_folds": INNER_FOLDS,
    "primary_metric": "macro-F1",
    "note": "Target encoded numerically for all classifiers; saved OOF labels decoded to cultivar names."
}
with open(LOG / "NB03_run_info.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2, ensure_ascii=False)

display(summary)
print("\nNB03 completed successfully.")
